# Context-I2W × Fashion-IQ 零样本复现（评估）

本 notebook **只做评估**：加载权重 + 在 FashionIQ 验证集上算 R@10 / R@50 + 与论文 Table 4 对照。
**不重跑训练。**

## 需要挂载的数据集

| 数据集 | 内容 | 何时需要 |
|---|---|---|
| `cir-repro-code` | 本 bundle（官方代码 + repro 脚本），**不含图片、不含权重** | 必需 |
| `context-i2w-ckpt` | `epoch_30.pt`，需同时含 `state_dict` 与 `state_dict_img2text` | 第 5 格起 |
| `fashioniq-val` | `fashion-iq/{json, image_splits, images}`，15,415 张 PNG | 二选一：有就挂，没有就走 4b/4c 从镜像建 |

## 怎么跑

- 第 **1–4** 格现在就能跑：验证代码、依赖、tokenizer 都正常，图片数据没到也不影响。
- 第 **4b / 4c** 格只在打算从镜像取图时才需要。
- 第 **5–7** 格等图片和权重到位后再跑。
- **Settings → Accelerator 选 GPU（T4 即可）**。
- **Internet 要打开**：第 2 格要装 `einops_exts`，第 4b 格要从 GitHub / HuggingFace 下载标注与镜像。
  第 5–7 格本身不联网（代码来自挂载的数据集，词表 `bpe_simple_vocab_16e6.txt.gz` 随代码打包）。

## 数字口径

论文 Table 4（zero-shot, attribute manipulation）：

| 类别 | R@10 | R@50 |
|---|---|---|
| Dress | 23.1 | 45.3 |
| Shirt | 29.7 | 48.6 |
| TopTee | 30.6 | 52.9 |
| **Average** | **27.8** | **48.9** |

## 1. 环境自检

In [ ]:
import sys, os, glob, platform
import torch

print("Python :", sys.version.split()[0])
print("torch  :", torch.__version__, "| cuda build:", torch.version.cuda)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU    : {p.name}  {p.total_memory / 1e9:.1f} GB  sm_{p.major}{p.minor}")
else:
    print("GPU    : 没有！去 Settings -> Accelerator 选 GPU")

print("\n/kaggle/input 下挂载的内容：")
for p in sorted(glob.glob("/kaggle/input/*")):
    n_files, n_bytes = 0, 0
    for root, _dirs, files in os.walk(p):
        n_files += len(files)
        for f in files:
            try:
                n_bytes += os.path.getsize(os.path.join(root, f))
            except OSError:
                pass
    print(f"  {p:<45} {n_files:>7} 个文件  {n_bytes / 1e9:>6.2f} GB")

## 2. 依赖

Kaggle 镜像自带 torch / torchvision / Pillow / numpy / pandas。
这里只补官方 `model/model.py` 用到的 `einops`、`einops_exts`，以及分词器需要的 `ftfy`、`regex`、`braceexpand`。

`third_party/open_clip/clip.py` 用了 `from pkg_resources import packaging`，新版 setuptools 里可能已经没有 `pkg_resources`，下面会检查并补一个最小替身。

In [ ]:
import importlib, subprocess, sys

NEED = ["einops", "einops_exts", "ftfy", "regex", "braceexpand", "tqdm"]
missing = []
for m in NEED:
    try:
        importlib.import_module(m)
        print(f"[OK]   {m}")
    except ImportError:
        print(f"[MISS] {m}")
        missing.append(m)

if missing:
    print("\n安装:", " ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input", *missing], check=True)
    for m in missing:
        importlib.import_module(m)
    print("安装完成")

# 官方 third_party/open_clip/clip.py 第 6 行: `from pkg_resources import packaging`
# 新版 setuptools 的 pkg_resources 不再暴露 packaging 属性，这一行会 ImportError
try:
    from pkg_resources import packaging  # noqa: F401
    print("[OK]   pkg_resources.packaging")
except Exception:
    import types
    import packaging, packaging.version  # noqa: F401
    try:
        import pkg_resources as _pr
        _pr.packaging = packaging
        print("[PATCH] 给 pkg_resources 补上 packaging 属性")
    except ImportError:
        shim = types.ModuleType("pkg_resources")
        shim.packaging = packaging
        sys.modules["pkg_resources"] = shim
        print("[PATCH] 注入 pkg_resources 替身")

## 3. 定位代码 / 数据 / 权重

不管你数据集叫什么名字、zip 有没有自动解压、目录套了几层，这里都按特征文件去搜：

- 代码：找同时含 `model/model.py`、`third_party/open_clip/clip.py`、`src/eval_utils.py` 的目录
- 数据：找含 `json/cap.dress.val.json` 的目录的上一层
- 权重：`/kaggle/input` 下最大的 `*.pt`

In [ ]:
import glob, os, zipfile
from pathlib import Path

# --- 兜底：如果数据集里的 zip 没被自动解压 ---
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    if "cir_repro" not in os.path.basename(z).lower():
        continue
    dest = Path("/kaggle/working/_unzipped")
    if not dest.exists():
        print("解压", z)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)

def find_repo():
    hits = []
    for p in glob.glob("/kaggle/input/**/model/model.py", recursive=True) + \
             glob.glob("/kaggle/working/_unzipped/**/model/model.py", recursive=True):
        repo = Path(p).parent.parent
        if (repo / "third_party" / "open_clip" / "clip.py").is_file() and (repo / "src" / "eval_utils.py").is_file():
            hits.append(repo)
    return sorted(hits, key=lambda x: len(x.parts))[0] if hits else None

def find_data_root():
    hits = []
    for pat in ("/kaggle/input/**/json/cap.dress.val.json",
                "/kaggle/working/**/json/cap.dress.val.json"):
        for p in glob.glob(pat, recursive=True):
            root = Path(p).parent.parent
            if (root / "images").is_dir() and (root / "image_splits").is_dir():
                hits.append(root)
    return sorted(hits, key=lambda x: len(x.parts))[0] if hits else None

def find_scripts():
    hits = []
    for pat in ("/kaggle/input/**/reproduce.py", "/kaggle/working/_unzipped/**/reproduce.py"):
        for p in glob.glob(pat, recursive=True):
            d = Path(p).parent
            if (d / "preflight.py").is_file() and (d / "probe_data_sources.py").is_file():
                hits.append(d)
    return sorted(hits, key=lambda x: len(x.parts))[0] if hits else None

def find_checkpoint():
    cands = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True), key=os.path.getsize, reverse=True)
    return Path(cands[0]) if cands else None

REPO = find_repo()
SCRIPTS = find_scripts()
DATA_ROOT = find_data_root()
CKPT = find_checkpoint()

print("代码  REPO     :", REPO or "未找到 —— 检查是否 Add Data 了 cir-repro-code")
print("脚本  SCRIPTS  :", SCRIPTS or "未找到 —— 代码包里缺 repro/ 目录")
print("数据  DATA_ROOT:", DATA_ROOT or "未找到（第 6-8 格需要）")
print("权重  CKPT     :", CKPT or "未找到（第 6-8 格需要）")
if CKPT:
    print(f"      大小      : {CKPT.stat().st_size / 1e9:.2f} GB")

# 权重结构检查：缺一个键就跑不起来
if CKPT:
    import torch
    try:
        ck = torch.load(CKPT, map_location="cpu", weights_only=False)
    except TypeError:
        ck = torch.load(CKPT, map_location="cpu")
    keys = list(ck.keys()) if isinstance(ck, dict) else []
    print("\n检查点顶层键:", keys)
    for k in ("state_dict", "state_dict_img2text", "epoch"):
        print(("  [OK]   " if k in ck else "  [缺失] ") + k)
    if "state_dict_img2text" in ck:
        wr = [k for k in ck["state_dict_img2text"] if "W_R" in k]
        print("  W_R 参数:", wr[:4] if wr else "找不到 —— 可能不是 Context-I2W 权重")
    del ck

## 4. 代码可导入性验证

数据集目录是**只读**的，先复制到 `/kaggle/working/cir`（约 11 MB）再 import，避免 `__pycache__` 写入报错。

这一格能跑通，就说明：模型代码、open_clip 分词器、离线词表、官方指标函数全部正常。

In [ ]:
import shutil, sys
from pathlib import Path

if REPO is None or SCRIPTS is None:
    raise SystemExit("找不到代码或脚本目录，先把 cir-repro-code 数据集 Add Data 进来")

WORK = Path("/kaggle/working/cir")
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(REPO, WORK)                              # 官方代码 → 工作目录根
shutil.copytree(SCRIPTS, WORK / "repro")                 # 我们的脚本 → WORK/repro
print("代码已复制到", WORK, "（含 repro/ 脚本）")

# Kaggle 会把数据集里的 .gz 自动解压，但官方分词器和 repro 脚本都是按 .gz 打开的
import gzip
for target in (WORK / "third_party/open_clip/bpe_simple_vocab_16e6.txt.gz",
               WORK / "repro/image_manifest.csv.gz"):
    if target.exists():
        continue
    plain = target.with_suffix("")          # 去掉 .gz 后的名字
    if not plain.is_file():
        continue
    with plain.open("rb") as fin, gzip.open(target, "wb") as fout:
        shutil.copyfileobj(fin, fout)
    print(f"  [修复] {plain.name} 被 Kaggle 解压了，已重新压成 {target.name}")

for p in (str(WORK), str(WORK / "src")):
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

import model.model as _mm
import third_party.open_clip.clip as _oc
from model.clip import _transform
from eval_utils import get_metrics_fashion

tok = int(_oc.tokenize(["<|replace|>"])[0, 1].item())
print("model.model          : OK  (build_model, IM2TEXT_OWM_Attn 可用)")
print("third_party.open_clip: OK  离线词表加载成功")
print("model.clip._transform: OK")
print("src.eval_utils       : OK  (get_metrics_fashion 可用)")
print("split token 位置     :", tok, "  <- 官方模板 'a photo of <|replace|> , ...' 的第 2 个 token")

# 确认官方 evaluate_fashion 的坑还在（提醒我们自己不走它）
import inspect
src = inspect.getsource(__import__("eval_utils").evaluate_fashion)
print("\nevaluate_fashion 内是否循环了 4 种特征:", "mixture" in src and "composed" in src)

## 4b. 数据来源探测（可选，只有打算从镜像取图时才需要）

FashionIQ 的图片**不在 GitHub 上**——GitHub 只有标注（`XiaoxiaoGuo/fashion-iq`）和图片 URL 映射（`hongwang600/fashion-iq-metadata`）。图片本身要另外获取。

这一格会：拉官方标注 → 下载 WISER 镜像（约 0.98 GB）→ 用成员 B 留下的 `image_manifest.csv.gz`（15,415 张图的逐张尺寸指纹）比对，判断镜像里的图**是不是同一批**。

- 尺寸全部一致 → 可以用镜像，报告里写明来源与校验方式
- 有尺寸差异或覆盖不全 → 用成员 B 那套图片，别用镜像

约 10–15 分钟。已经挂载了现成图片数据集的话，这一格不用跑。

In [ ]:
import subprocess, sys

probe = WORK / "repro" / "probe_data_sources.py"
manifest = WORK / "repro" / "image_manifest.csv.gz"

if DATA_ROOT is not None:
    print("已经找到现成的图片数据集，跳过探测：", DATA_ROOT)
elif not probe.is_file() or not manifest.is_file():
    print("代码包里没有 probe_data_sources.py / image_manifest.csv.gz，"
          "说明挂载的 cir-repro-code 还是旧版本，请重新 Add Data 选择最新版本。")
else:
    subprocess.run([sys.executable, "-u", str(probe),
                    "--work-dir", "/kaggle/working/probe",
                    "--manifest", str(manifest)])

## 4c. 从镜像构建数据集（4b 判定通过后才跑）

**只有 4b 的结论是 `match_on_dimensions` 才应该跑这一格。**

做的事：把镜像解压出来的原图统一转成 PNG（官方 `src/data.py` 写死 `<id>.png`），
连同官方标注一起排成

```
/kaggle/working/fashion-iq/
  fashion-iq/
    json/cap.{dress,shirt,toptee}.val.json
    image_splits/split.{dress,shirt,toptee}.val.json
    images/<image_id>.png        15,415 张
  prepare_report.json
```

如果镜像里本来就是 PNG 且与审计集逐字节相同，会直接复制而不是重编码。

约 5–12 分钟。可重复执行，已转好的会跳过。跑完这一格 `DATA_ROOT` 会自动重新定位，
第 5–7 格直接往下跑即可。

In [ ]:
import subprocess, sys
from pathlib import Path

if DATA_ROOT is not None:
    print("已有现成数据，跳过构建：", DATA_ROOT)
elif not (WORK / "repro" / "prepare_data.py").is_file():
    print("代码包里没有 prepare_data.py，说明挂载的 cir-repro-code 还是旧版本，请重新 Add Data。")
else:
    r = subprocess.run([sys.executable, "-u", str(WORK / "repro" / "prepare_data.py"),
                        "--work-dir", "/kaggle/working/probe",
                        "--manifest", str(WORK / "repro" / "image_manifest.csv.gz"),
                        "--out", "/kaggle/working/fashion-iq"])
    print("\n退出码:", r.returncode, "（0 = 数据集可用）")
    DATA_ROOT = find_data_root()
    print("重新定位 DATA_ROOT:", DATA_ROOT)

## 5. 前置检查（数据和权重到位后再跑）

逐张解码 15,415 张图，并核对查询数、图库覆盖、权重结构与 SHA256。
**有阻断性问题会直接失败**——这正是官方代码会静默跳过、把指标分母悄悄改小的那种错误。

约 1–3 分钟。

In [ ]:
import subprocess, sys
from pathlib import Path

if DATA_ROOT is None or CKPT is None:
    print("数据或权重还没挂载，先跑第 1-4 格验证代码即可。")
else:
    out = Path("/kaggle/working/repro_out/preflight.json")
    out.parent.mkdir(parents=True, exist_ok=True)
    r = subprocess.run([sys.executable, "-u", str(WORK / "repro" / "preflight.py"),
                        "--data-root", str(DATA_ROOT), "--checkpoint", str(CKPT),
                        "--out", str(out)])
    print("\n退出码:", r.returncode, "（0 = 可以开跑）")

## 6. 小规模自检（约 1 分钟，**数字不可用于报告**）

shirt 类别，32 条查询 / 256 张图库。用来暴露路径、权重结构、显存、模板等运行时问题。

如果这一格报错就**不要**往下跑全量，把错误贴出来。

In [ ]:
import subprocess, sys

if DATA_ROOT is None or CKPT is None:
    print("数据或权重还没挂载，跳过。")
else:
    subprocess.run([sys.executable, "-u", str(WORK / "repro" / "reproduce.py"),
                    "--repo", str(WORK),
                    "--data-root", str(DATA_ROOT),
                    "--checkpoint", str(CKPT),
                    "--categories", "shirt",
                    "--max-queries", "32", "--max-gallery", "256",
                    "--batch-size", "32", "--workers", "2",
                    "--out", "/kaggle/working/repro_out/smoke"], check=True)

## 7. 全量复现（三个类别 × 完整验证集，**报告用的就是这一格**）

T4 上约 20–40 分钟。显存不够就把 `--batch-size` 降到 32。

同时会把图库与查询特征存到 `repro_out/features/`，后面做属性反馈 + IVS 全库重排可以直接复用，不必再编码一遍 15,415 张图。

In [ ]:
import subprocess, sys

if DATA_ROOT is None or CKPT is None:
    print("数据或权重还没挂载，跳过。")
else:
    subprocess.run([sys.executable, "-u", str(WORK / "repro" / "reproduce.py"),
                    "--repo", str(WORK),
                    "--data-root", str(DATA_ROOT),
                    "--checkpoint", str(CKPT),
                    "--categories", "dress,shirt,toptee",
                    "--batch-size", "64", "--workers", "4",
                    "--out", "/kaggle/working/repro_out"], check=True)

## 8. 论文对照

In [ ]:
import json
import pandas as pd

p = "/kaggle/working/repro_out/summary.json"
s = json.load(open(p, encoding="utf-8"))

print("权重 :", s["checkpoint"]["path"], "| epoch", s["checkpoint"]["epoch"])
print("指标函数:", s["metric_function"])
print("环境 :", s["environment"]["gpu"], "| torch", s["environment"]["torch"])

rows = []
for cat in ["dress", "shirt", "toptee", "average"]:
    c = s["comparison"].get(cat)
    if not c:
        continue
    rows.append({
        "类别": cat,
        "R@10 论文": c["paper_R@10"], "R@10 复现": c["repro_R@10"], "Δ10": c["diff_R@10"],
        "R@50 论文": c["paper_R@50"], "R@50 复现": c["repro_R@50"], "Δ50": c["diff_R@50"],
    })

df = pd.DataFrame(rows).set_index("类别")
display(df)
print("\n各项与论文的差在 ±0.5 以内 = 复现成功；普遍低 1-3 个点 = 图片版本差异（见 README 第 6 节）")

## 9. 产出的文件

```
/kaggle/working/repro_out/
  preflight.json     数据与权重逐项体检（含权重 SHA256，报告里举证"同一权重"用）
  summary.json       复现指标 + 环境 + 论文对照 —— 报告里所有数字都必须来自它
  features/          图库/查询特征、目标名次，供后续全库重排复用
```

跑完记得在 Kaggle 右上角 **Save Version**，否则 `/kaggle/working` 会丢。